# Module B Implementation and Optimization Report

> CS 432 Databases - Assignment 2 (Module B)

This report documents implementation evidence for SubTask 1 to SubTask 5:
- SubTask 1: Local environment setup and data integrity
- SubTask 2: Session-validated API and UI
- SubTask 3: RBAC and security logging
- SubTask 4: SQL indexing and query optimization
- SubTask 5: Before/after benchmarking and EXPLAIN analysis

## 1) Schema Design and Data Integrity (SubTask 1)

Core/project separation used in this module:
- Core identity/auth tables: `Member`, `AuthCredential`
- Core relation tables: `Follow`, `GroupMember`, `Group`
- Project content tables: `Post`, `Comment`

Integrity design choices:
- Credentials are not duplicated in project tables (stored only in `AuthCredential`).
- Foreign keys with cascade policies maintain consistency during create/delete operations.
- Constraint and trigger rules enforce data validity and business rules.

Member lifecycle requirement coverage:
- Admin member creation writes to both `Member` and `AuthCredential`.
- Member deletion cascades through related mappings/content as defined by schema constraints.

## 2) Security, Session Validation, and RBAC (SubTask 2 and 3)

Session validation:
- All protected endpoints validate a local JWT session token (`session-token` header).
- Invalid, missing, or expired sessions return `401`.

RBAC model:
- Admin users: member management, group membership administration, log inspection.
- Regular users: can modify only their own portfolio/posts/comments.
- Unauthorized cross-user modification attempts return `403`.

Security logging:
- API audit file: `Module_B/logs/audit.log`.
- DB-side write tracking: `ApiWriteLog` + triggers classify writes as `API` or `DIRECT_DB`.
- This makes direct SQL updates outside API/session validation easily identifiable as unauthorized.

## 3) SQL Indexing and Query Optimization (SubTask 4)

Frequently accessed / benchmarked endpoints:
- `GET /posts` (feed listing with filtering and ordering)
- `GET /posts/{post_id}/comments` (hotspot query under high-comment post)

Indexing strategy applied to API query clauses:
1. `idx_post_active_postdate ON Post(IsActive, PostDate DESC)`
   - Targets: `WHERE p.IsActive = TRUE` + `ORDER BY p.PostDate DESC`
2. `idx_comment_post_active_date ON Comment(PostID, IsActive, CommentDate ASC)`
   - Targets: `WHERE c.PostID = ? AND c.IsActive = TRUE` + `ORDER BY c.CommentDate ASC`

Both are project-table indexes aligned to actual WHERE/JOIN/ORDER BY patterns.

## 4) Performance Benchmarking and EXPLAIN Evidence (SubTask 5)

Methodology:
- Captured SQL execution times and API response times for the same endpoints.
- Executed two stages with identical benchmark parameters:
  - `before_indexes`
  - `after_indexes`
- Captured EXPLAIN plans in both stages and compared access characteristics (`type`, `key`, `rows`, `extra`).

Reproducibility steps:
1. Start API server:
   - `cd Module_B/app`
   - `uvicorn main:app --reload --port 8001`
2. Run benchmark script:
   - `cd Module_B/app`
   - `python run_index_benchmark.py`
3. Inspect artifact:
   - `Module_B/performance/index_benchmark_results.json`

Interpretation note:
- Speedups may vary run-to-run depending on dataset size/distribution and cache effects.
- EXPLAIN output is the primary evidence of access-plan changes.

In [4]:
import json
from pathlib import Path

candidate_paths = [
    Path("performance/index_benchmark_results.json"),
    Path("index_benchmark_results.json"),
    Path("Module_B/performance/index_benchmark_results.json"),
]

results_path = next((p for p in candidate_paths if p.exists()), None)
if results_path is None:
    raise FileNotFoundError(
        "Could not find index_benchmark_results.json. Run Module_B/app/run_index_benchmark.py first."
    )

with results_path.open("r", encoding="utf-8") as f:
    results = json.load(f)

print(f"Loaded benchmark file: {results_path}")
results["benchmark_params"]

Loaded benchmark file: performance\index_benchmark_results.json


{'iterations': 40,
 'warmup': 5,
 'limit': 20,
 'offset': 1510,
 'comment_post_id': 3021}

In [5]:
before = results["stages"][0]
after = results["stages"][1]
speedup = results["speedup"]

summary = {
    "before_sql_posts_avg_ms": before["sql_ms"]["list_posts"]["avg_ms"],
    "after_sql_posts_avg_ms": after["sql_ms"]["list_posts"]["avg_ms"],
    "before_sql_comments_avg_ms": before["sql_ms"]["list_comments"]["avg_ms"],
    "after_sql_comments_avg_ms": after["sql_ms"]["list_comments"]["avg_ms"],
    "before_api_posts_avg_ms": before["api_ms"]["list_posts"]["avg_ms"],
    "after_api_posts_avg_ms": after["api_ms"]["list_posts"]["avg_ms"],
    "before_api_comments_avg_ms": before["api_ms"]["list_comments"]["avg_ms"],
    "after_api_comments_avg_ms": after["api_ms"]["list_comments"]["avg_ms"],
    "posts_sql_speedup": speedup["posts_sql_speedup"],
    "comments_sql_speedup": speedup["comments_sql_speedup"],
    "posts_api_speedup": speedup["posts_api_speedup"],
    "comments_api_speedup": speedup["comments_api_speedup"],
}

summary

{'before_sql_posts_avg_ms': 5.601,
 'after_sql_posts_avg_ms': 4.877,
 'before_sql_comments_avg_ms': 24.173,
 'after_sql_comments_avg_ms': 19.638,
 'before_api_posts_avg_ms': 33.422,
 'after_api_posts_avg_ms': 37.511,
 'before_api_comments_avg_ms': 96.232,
 'after_api_comments_avg_ms': 97.535,
 'posts_sql_speedup': 1.148,
 'comments_sql_speedup': 1.231,
 'posts_api_speedup': 0.891,
 'comments_api_speedup': 0.987}

In [6]:
def compact_explain(rows):
    return [{
        "table": r.get("table"),
        "type": r.get("type"),
        "key": r.get("key"),
        "rows": r.get("rows"),
        "extra": r.get("extra"),
    } for r in rows]

print("Before EXPLAIN (posts):")
for row in compact_explain(before["explain"]["list_posts"]):
    print(row)

print("\nAfter EXPLAIN (posts):")
for row in compact_explain(after["explain"]["list_posts"]):
    print(row)

print("\nBefore EXPLAIN (comments):")
for row in compact_explain(before["explain"]["list_comments"]):
    print(row)

print("\nAfter EXPLAIN (comments):")
for row in compact_explain(after["explain"]["list_comments"]):
    print(row)

Before EXPLAIN (posts):
{'table': 'p', 'type': 'ALL', 'key': None, 'rows': 3021, 'extra': 'Using where; Using filesort'}
{'table': 'm', 'type': 'eq_ref', 'key': 'PRIMARY', 'rows': 1, 'extra': None}

After EXPLAIN (posts):
{'table': 'p', 'type': 'ALL', 'key': None, 'rows': 3021, 'extra': 'Using where; Using filesort'}
{'table': 'm', 'type': 'eq_ref', 'key': 'PRIMARY', 'rows': 1, 'extra': None}

Before EXPLAIN (comments):
{'table': 'c', 'type': 'ref', 'key': 'idx_comment_post', 'rows': 2500, 'extra': 'Using where; Using filesort'}
{'table': 'm', 'type': 'eq_ref', 'key': 'PRIMARY', 'rows': 1, 'extra': None}

After EXPLAIN (comments):
{'table': 'c', 'type': 'range', 'key': 'idx_comment_post_active_date', 'rows': 2500, 'extra': 'Using index condition'}
{'table': 'm', 'type': 'eq_ref', 'key': 'PRIMARY', 'rows': 1, 'extra': None}


## 5) Video Demonstration Link

Add your hosted 3-5 minute demo link here (YouTube unlisted or open Drive link):
- `<PASTE_VIDEO_LINK_HERE>`

## 6) Conclusion

- Module B integrates secure local APIs, RBAC enforcement, and auditable DB operations.
- Indexing/benchmarking pipeline is implemented with before-vs-after quantitative evidence.
- Remaining result analysis can be expanded with larger datasets if needed.